# 🌟 CAROLINA AI • SUPERGPU EDITION (GOOGLE COLAB)
### 🚀 Supercomputadora de IA con NVIDIA GPU • 16 GB RAM • Blender 3D • Manim 4K • Telegram Cloud

---

## 1️⃣ Paso 1: Instalar Dependencias (Blender, Manim, Cloudflared y Túnel)

In [ ]:
import os, sys

print("📦 1/3: Instalando herramientas del sistema (Blender 3D, FFmpeg, Node)...")
!apt-get update -qq && apt-get install -y -qq blender ffmpeg curl libcairo2-dev libpango1.0-dev nodejs npm > /dev/null 2>&1

print("⚡ 2/3: Instalando Cloudflared y Localtunnel...")
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
!npm install -g localtunnel -s > /dev/null 2>&1

print("🐍 3/3: Instalando librerías Python...")
!pip install -q manim imageio-ffmpeg yt-dlp sympy duckdb pandas plotly matplotlib beautifulsoup4 requests rich pypdf python-docx openpyxl psutil

print("\n🎉 ¡Instalación completada!")

## 2️⃣ Paso 2: Descargar Carolina AI

In [ ]:
import os
if not os.path.exists('/content/carolina_ai'):
    !git clone https://github.com/miguel151100/caroli.a.git /content/carolina_ai
else:
    !cd /content/carolina_ai && git pull

os.makedirs('/content/ALMACENAMIENTO_CAROLINA/MAC', exist_ok=True)
os.makedirs('/content/ALMACENAMIENTO_CAROLINA/LINUX', exist_ok=True)
print("✅ Carolina AI lista.")

## 3️⃣ Paso 3: 🚀 Encender Servidor y Generar Tu Enlace

In [ ]:
import os, time, re, subprocess
from IPython.display import display, HTML

# 1. Detener procesos anteriores
!pkill -9 -f Claude_Pro_App.py || true
!pkill -9 -f cloudflared || true
!pkill -9 -f lt || true
!rm -f /content/tunnel.log /content/lt.log
time.sleep(1)

# 2. Iniciar Carolina AI en puerto 5055
print("🌟 Iniciando servidor Carolina AI en puerto 5055...")
!nohup python3 /content/carolina_ai/Claude_Pro_App.py > /content/server.log 2>&1 &
time.sleep(3)

# 3. Iniciar Cloudflare Tunnel con protocolo http2 (indispensable en Colab porque Colab bloquea UDP)
print("🌍 Creando túnel seguro HTTPS...")
!nohup cloudflared tunnel --protocol http2 --url http://127.0.0.1:5055 --logfile /content/tunnel.log > /dev/null 2>&1 &
!nohup lt --port 5055 > /content/lt.log 2>&1 &

tunnel_url = None
for _ in range(20):
    time.sleep(1)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r", errors="ignore") as f:
            txt = f.read()
            m = re.search(r'(https://[a-zA-Z0-9-]+\.trycloudflare\.com)', txt)
            if m:
                tunnel_url = m.group(1)
                break

# Si Cloudflare tardara, buscar en localtunnel como respaldo
backup_url = None
if os.path.exists("/content/lt.log"):
    with open("/content/lt.log", "r", errors="ignore") as f:
        m2 = re.search(r'(https://[a-zA-Z0-9-]+\.loca\.lt)', f.read())
        if m2:
            backup_url = m2.group(1)

url_final = tunnel_url or backup_url

if url_final:
    print("\n" + "="*60)
    print(f"🎉 ¡AQUÍ ESTÁ TU ENLACE DE CAROLINA CON GPU EN COLAB!:")
    print(f"👉 {url_final}")
    print("="*60 + "\n")
    
    html = f"""
    <div style="background:linear-gradient(135deg,#0F172A,#1E293B);border:2px solid #38BDF8;border-radius:12px;padding:22px;color:#FFF;max-width:600px;font-family:sans-serif;">
      <h2 style="color:#38BDF8;margin-top:0;">🌟 Carolina AI • SuperGPU Activa</h2>
      <p>Tu servidor con GPU NVIDIA y Blender 3D está listo. Toca el botón para abrirlo:</p>
      <div style="text-align:center;margin:18px 0;">
        <a href="{url_final}" target="_blank" style="background:#2563EB;color:#FFF;padding:14px 32px;font-size:1.1rem;font-weight:bold;border-radius:8px;text-decoration:none;display:inline-block;">
          🚀 ABRIR CAROLINA AI
        </a>
      </div>
      <div style="background:#000;padding:10px;border-radius:6px;font-family:monospace;color:#38BDF8;text-align:center;">
        {url_final}
      </div>
    </div>
    """
    display(HTML(html))
else:
    print("⚠️ Buscando enlace en tunnel.log...")
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log") as f: print(f.read())

# Mantener vivo
while True:
    time.sleep(60)
